[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kulinskij-tech/Synergetics/blob/main/syn_py/syn_10_stochastic_synergetics.ipynb)


# 10. Noise, Fokker–Planck dynamics, and switching

_Graduate course: Introduction to Synergetics · English draft_

> **Guiding question:** When does noise reveal, shift, or create macroscopic behavior near a transition?

**Notebook contract.** Run from top to bottom. All parameters are dimensionless unless stated otherwise. Explain every diagnostic you use; a visually attractive trajectory is not by itself evidence of self-organization.


## Learning outcomes

By the end of this module, you should be able to:

- define a Markov process, Wiener increment, and Itô stochastic differential equation
- use quadratic variation to explain the correction term in Itô's formula
- translate between an overdamped Langevin equation and its Fokker–Planck equation
- validate Euler–Maruyama against the exact Ornstein–Uhlenbeck moments
- derive the stationary density for gradient drift
- estimate switching statistics and recognize discretization bias


## Probability language: trajectory, ensemble, and Markov state

A random variable $X$ assigns a number to each outcome of an experiment. A **stochastic process** $X_t$ is a time-indexed family of random variables. One realization $t\mapsto X_t$ is a sample path; repeating the experiment produces an ensemble of paths. The probability density $p(x,t)$ answers an ensemble question, whereas a time series is one path. Replacing an ensemble average by a long-time average requires an additional ergodicity argument.

A process is **Markovian** if, once the present state is specified, its conditional future distribution does not require the earlier history:

$$p(x_3,t_3\mid x_2,t_2;x_1,t_1)=p(x_3,t_3\mid x_2,t_2),\qquad t_1<t_2<t_3.$$

Its transition densities compose through the Chapman–Kolmogorov equation,

$$p(x_3,t_3\mid x_1,t_1)=\int p(x_3,t_3\mid x_2,t_2)p(x_2,t_2\mid x_1,t_1)\,dx_2.$$

This composition law is the probabilistic starting point for kinetic equations. A short-time expansion in conditional jump moments gives the Kramers–Moyal series; retaining drift and diffusion terms gives the Fokker–Planck equation.


## Wiener motion and the meaning of an Itô equation

A standard Wiener process $W_t$ has $W_0=0$, independent Gaussian increments, and

$$\Delta W=W_{t+\Delta t}-W_t\sim\mathcal N(0,\Delta t).$$

Therefore $\Delta W$ is typically of order $\sqrt{\Delta t}$, not $\Delta t$. Its paths are continuous but nowhere classically differentiable; “white noise” $\xi(t)=dW_t/dt$ is only a formal notation. The Itô stochastic differential equation

$$dX_t=a(X_t,t)\,dt+b(X_t,t)\,dW_t$$

means an integral equation whose random integrand is evaluated at the beginning of each time interval. Here $a$ is the drift and $b^2$ is the local variance rate. The simulation rule is Euler–Maruyama,

$$X_{n+1}=X_n+a(X_n,t_n)\Delta t+b(X_n,t_n)\sqrt{\Delta t}\,Z_n,
\qquad Z_n\sim\mathcal N(0,1).$$

A common error is to multiply the Gaussian draw by $\Delta t$; that makes the accumulated noise vanish in the continuum limit.


## Why ordinary calculus changes: quadratic variation and Itô's formula

Since $(\Delta W)^2$ is of order $\Delta t$, the second-order term in a Taylor expansion survives. In Itô bookkeeping,

$$(dW)^2=dt,\qquad dW\,dt=0,\qquad (dt)^2=0.$$

If $dX=a\,dt+b\,dW$ and $F=F(X,t)$, then

$$dF=\left(\partial_tF+a\,\partial_xF+\frac12b^2\partial_x^2F\right)dt
+b\,\partial_xF\,dW.$$

For the simplest example, $X=W$ and $F=X^2$, so $d(W^2)=2W\,dW+dt$. The extra $dt$ is not optional: it produces $\mathbb E[W_t^2]=t$.

With state-dependent noise, the convention matters. A Stratonovich equation $dX=a_Sdt+b\circ dW$ is equivalent to the Itô equation with $a_I=a_S+\tfrac12b\,\partial_xb$. For additive noise ($b=$ constant), the two interpretations coincide. Every model with multiplicative noise must state its convention and physical limiting procedure.


## From sample paths to density evolution

Applying Itô's formula to an arbitrary smooth test function and integrating by parts gives the forward Fokker–Planck equation

$$\partial_tp(x,t)=-\partial_x[a(x,t)p(x,t)]
+\frac12\partial_x^2[b^2(x,t)p(x,t)].$$

This is a conservation law $\partial_tp=-\partial_xJ$ with probability current

$$J=a p-\frac12\partial_x(b^2p).$$

Initial and boundary conditions are part of the model: reflecting boundaries impose $J=0$, while absorbing boundaries impose $p=0$. Normalization, positivity, and agreement of density moments with path ensembles are essential numerical checks. The SDE and Fokker–Planck equation are two descriptions of the same Markov diffusion: one evolves random paths, the other evolves their probability density.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.2, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(20260904)

# Ornstein-Uhlenbeck process: the basic relaxation-plus-noise benchmark.
gamma_ou, D_ou, dt_ou, t_end_ou = 1.2, 0.35, 0.005, 6.0
steps_ou = int(t_end_ou/dt_ou)
paths_ou, x0_ou = 5000, 1.5
x_ou = np.full(paths_ou, x0_ou)
mean_ou = np.empty(steps_ou + 1)
var_ou = np.empty(steps_ou + 1)
mean_ou[0], var_ou[0] = x_ou.mean(), x_ou.var()
sample_ou = np.empty((8, steps_ou + 1))
sample_ou[:, 0] = x_ou[:8]

for n in range(steps_ou):
    x_ou += -gamma_ou*x_ou*dt_ou + np.sqrt(2*D_ou*dt_ou)*rng.standard_normal(paths_ou)
    mean_ou[n+1], var_ou[n+1] = x_ou.mean(), x_ou.var()
    sample_ou[:, n+1] = x_ou[:8]

t_ou = np.arange(steps_ou + 1)*dt_ou
mean_exact = x0_ou*np.exp(-gamma_ou*t_ou)
var_exact = D_ou/gamma_ou*(1-np.exp(-2*gamma_ou*t_ou))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.7))
ax[0].plot(t_ou, sample_ou.T, lw=0.6)
ax[0].set(xlabel="time", ylabel="X", title="Individual OU paths")
ax[1].plot(t_ou, mean_ou, label="ensemble")
ax[1].plot(t_ou, mean_exact, "k--", label="exact")
ax[1].set(xlabel="time", ylabel="mean", title="Relaxation of the mean")
ax[1].legend()
ax[2].plot(t_ou, var_ou, label="ensemble")
ax[2].plot(t_ou, var_exact, "k--", label="exact")
ax[2].set(xlabel="time", ylabel="variance", title="Noise builds stationary variance")
ax[2].legend()
plt.tight_layout()


## Noisy order-parameter dynamics

For the Itô process

$$dx=-V'(x)dt+\sqrt{2D}\,dW_t,$$

the density evolves as

$$\partial_tp=\partial_x[V'(x)p]+D\partial_x^2p.$$

Under zero probability current, $p_s(x)\propto e^{-V(x)/D}$. With $V=x^4/4-ax^2/2-hx$, the two wells represent competing macrostates. Weak noise produces rare barrier crossings; increasing $D$ broadens local fluctuations and accelerates switching.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.2, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(20260904)

a, h, D = 1.0, 0.05, 0.10
dt, steps = 0.005, 400000
x = np.empty(steps)
x[0] = -1.0
for n in range(steps-1):
    drift = a*x[n] - x[n]**3 + h
    x[n+1] = x[n] + drift*dt + np.sqrt(2*D*dt)*rng.standard_normal()

grid = np.linspace(-2.0, 2.0, 500)
V = grid**4/4 - a*grid**2/2 - h*grid
p = np.exp(-(V-V.min())/D)
p /= np.trapezoid(p, grid)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(np.arange(25000)*dt, x[:25000], lw=0.6)
ax[0].set(xlabel="time", ylabel="x", title="Noisy bistable trajectory")
ax[1].hist(x[50000:], bins=90, density=True, alpha=0.6, label="simulation")
ax[1].plot(grid, p, "k", lw=2, label=r"$e^{-V/D}/Z$")
ax[1].set(xlabel="x", ylabel="probability density", title="Stationary distribution")
ax[1].legend()
plt.tight_layout()


## What to measure

A histogram tests the stationary measure but discards temporal information. Switching analysis needs an operational state definition with hysteretic thresholds, residence-time distributions, and uncertainty estimates. Euler–Maruyama has finite-step bias; repeat with smaller $dt$ before interpreting discrepancies as physics.


## Assessed exercise

Part A: repeat the Ornstein–Uhlenbeck simulation at three time steps. Compare the empirical mean and variance with the exact formulas and determine the weak convergence trend. Part B: for at least four noise intensities in the bistable model, estimate mean residence times using two-threshold state detection. Test the Arrhenius/Kramers expectation by plotting $\log\tau$ versus $1/D$, and identify where the asymptotic relation fails.

Your submission must include a claim, the mathematical or physical reason for it, numerical evidence, and one limitation.


## Reading and attribution

**Local course reading:** The local *Nonequilibrium Statistical Physics / Physical Kinetics* notes, sections on Markov processes, Chapman–Kolmogorov, Langevin dynamics, Fokker–Planck, and Itô–Stratonovich calculus; Haken, *Synergetics: An Introduction*, fluctuations and nonequilibrium transitions.

**Verified web reading:**
- [Gardiner, Stochastic Methods](https://link.springer.com/book/9783540707127)
- [Risken, The Fokker–Planck Equation](https://doi.org/10.1007/978-3-642-61544-3)
- [van Kampen, Stochastic Processes in Physics and Chemistry](https://shop.elsevier.com/books/stochastic-processes-in-physics-and-chemistry/van-kampen/978-0-444-52965-7)
- [Haken's Synergetics: An Introduction](https://doi.org/10.1007/978-3-642-88338-5)

The local books are course-provided sources. Web links are supplementary and were verified on 2026-09-04.
